# 02. Baseline cho bốn chuỗi

**Mục tiêu:** chạy bốn baseline (Last Value, Moving Average 3 giờ, Seasonal Naive 24 giờ, Seasonal Naive 168 giờ)
cho **cả bốn chuỗi** của nhóm (chuỗi chính MT_158 và ba chuỗi phụ), cùng một quy trình, để có mốc so sánh cho MLP và LSTM
và có bảng tổng hợp bốn chuỗi cho Chương 3.

**Quy ước (giữ nguyên từ Nhiệm vụ 1 và 2):**
- Dữ liệu là kW trung bình mỗi giờ, nhãn thời gian là mốc kết thúc của giờ.
- Chia theo thời gian: 27.720 / 3.672 / 3.672 giờ. Ngữ cảnh lịch sử 168 giờ lấy từ chuỗi đầy đủ, nhưng **nhãn chỉ lấy trong tập đích**,
  nên mọi baseline đều được chấm trên đúng 3.672 mốc.
- MASE dùng mẫu số naive-1 tính trên tập train (`d_train`).
- Chọn baseline tốt nhất chỉ dựa trên **validation**; test chỉ để báo cáo.

**Cách chạy:** mở notebook trong thư mục `notebooks/` hoặc ở thư mục gốc, chạy lần lượt từ trên xuống.
Cần `data/processed/series_hourly.csv` (đã có trong kho). Danh sách chuỗi đọc từ `configs/config.yaml`.
Kết quả ghi vào `results/` với hậu tố tên chuỗi (ví dụ `baseline_results_MT_158.csv`).

In [ ]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.baseline import BASELINE_COLUMNS, get_common_eval_timestamps, predict_baselines
from src.config import load_config, path_from, set_seed
from src.data import load_hourly_data, split_by_time
from src.dst_check import dst_timestamps
from src.metrics import calculate_d_train, evaluate_by_group, evaluate_forecast

cfg = load_config()
set_seed(cfg["project"]["seed"])

MAIN = cfg["customers"]["main"]
CHAINS = [MAIN] + [c for c in cfg["customers"]["hourly_file_columns"] if c != MAIN]   # chuỗi chính đứng đầu

FIG_DIR = path_from(cfg, "figures_dir"); FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR = path_from(cfg, "tables_dir");  TAB_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR = ROOT / "results";              RES_DIR.mkdir(exist_ok=True)
MODEL_DIR = path_from(cfg, "models_dir"); MODEL_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
print("Chuỗi chính:", MAIN, "| Các chuỗi chạy:", CHAINS)

## 1. Đọc dữ liệu và chia tập

`load_hourly_data` kiểm tra dữ liệu (không trùng, không thiếu, không âm, cách đều 1 giờ) và dừng nếu sai.
`split_by_time` kiểm tra số dòng từng tập đúng 27.720 / 3.672 / 3.672.

In [ ]:
full_df = load_hourly_data(cfg, customers=CHAINS)
train_df, val_df, test_df = split_by_time(full_df, cfg)

print(f"Tổng: {len(full_df):,} giờ ({full_df.index.min()} -> {full_df.index.max()})")
print(f"Train: {len(train_df):,} | Validation: {len(val_df):,} | Test: {len(test_df):,} giờ")
full_df.describe().round(2)

## 2. Tính `d_train` và chuẩn hóa

`d_train` là trung bình |y_t − y_(t−1)| trên **tập train** của từng chuỗi, dùng làm mẫu số của MASE.

Bộ chuẩn hóa `StandardScaler` được khớp **chỉ trên tập train**, **mỗi chuỗi một bộ riêng** và lưu thành
`models/scaler_<tên chuỗi>.pkl` để mô hình học sâu khôi phục đúng thang đo gốc. Baseline vẫn dự báo trên giá trị kW gốc,
nên scaler ở đây chỉ để bàn giao cho Hồng Sơn và Bảo Quốc.

In [ ]:
d_train = {}
scaler_info = []
for c in CHAINS:
    d_train[c] = calculate_d_train(train_df[c].to_numpy(dtype=float))
    if not np.isfinite(d_train[c]) or d_train[c] <= 0:
        raise ValueError(f"d_train không hợp lệ cho {c}: {d_train[c]}")

    scaler = StandardScaler().fit(train_df[[c]].to_numpy())   # ép 1 cột, chỉ trên train
    joblib.dump(scaler, MODEL_DIR / f"scaler_{c}.pkl")
    scaler_info.append({"customer": c, "d_train": d_train[c],
                        "scaler_mean": scaler.mean_[0], "scaler_scale": scaler.scale_[0]})

pd.DataFrame(scaler_info).set_index("customer").round(4)

## 3. Mốc thời gian đánh giá

`get_common_eval_timestamps` giữ những mốc có đủ 168 giờ lịch sử, tính trên chuỗi đầy đủ. Vì ngữ cảnh được lấy từ cuối tập liền trước,
cả validation và test đều giữ đủ 3.672 mốc. Phần kiểm tra bên dưới xác nhận danh sách mốc trùng khít index của từng tập.

In [ ]:
val_ts, test_ts = {}, {}
for c in CHAINS:
    full_series = full_df[c].astype(float)
    val_ts[c] = get_common_eval_timestamps(full_series, val_df.index)
    test_ts[c] = get_common_eval_timestamps(full_series, test_df.index)

    assert val_ts[c].equals(val_df.index), f"{c}: mốc validation lệch index tập"
    assert test_ts[c].equals(test_df.index), f"{c}: mốc test lệch index tập"
    print(f"{c}: validation {len(val_ts[c]):,} mốc, test {len(test_ts[c]):,} mốc (khớp index tập)")

## 4. Dự báo và đánh giá trên validation

Dự báo cho từng mốc chỉ dùng giá trị **trước** mốc đó (Last Value: t−1; MA3: t−3 đến t−1; Seasonal Naive: t−24 hoặc t−168).
Mô hình tốt nhất được chọn theo MAE trên validation.

In [ ]:
val_pred, val_res, best_model = {}, {}, {}
for c in CHAINS:
    full_series = full_df[c].astype(float)
    pred = predict_baselines(full_series, val_ts[c])
    y_true = pred["y_true"].to_numpy(dtype=float)

    metrics = {}
    for col in BASELINE_COLUMNS:
        y_pred = pred[col].to_numpy(dtype=float)
        if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
            raise ValueError(f"{c}: có NaN/Inf ở {col}")
        metrics[col] = evaluate_forecast(y_true, y_pred, d_train=d_train[c])

    res = pd.DataFrame(metrics).T
    res.index.name = "model"
    val_pred[c], val_res[c] = pred, res.sort_values("MAE")
    best_model[c] = val_res[c].index[0]

    print(f"\n=== {c}: KẾT QUẢ VALIDATION (n = {len(y_true):,}) ===")
    print(val_res[c].round(4).to_string())
    print(f"-> Mô hình tốt nhất theo MAE validation: {best_model[c]}")

## 5. Đánh giá trên test

Test chỉ dùng để **báo cáo** cả bốn baseline, không dùng để chọn mô hình. Mô hình được xem là "tốt nhất" vẫn là mô hình đã chọn ở validation.

In [ ]:
test_pred, test_res = {}, {}
for c in CHAINS:
    full_series = full_df[c].astype(float)
    pred = predict_baselines(full_series, test_ts[c])
    y_true = pred["y_true"].to_numpy(dtype=float)

    metrics = {}
    for col in BASELINE_COLUMNS:
        y_pred = pred[col].to_numpy(dtype=float)
        if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
            raise ValueError(f"{c}: có NaN/Inf ở test, cột {col}")
        metrics[col] = evaluate_forecast(y_true, y_pred, d_train=d_train[c])

    res = pd.DataFrame(metrics).T
    res.index.name = "model"
    test_pred[c], test_res[c] = pred, res

    print(f"\n=== {c}: KẾT QUẢ TEST (n = {len(y_true):,}) ===")
    print(res.round(4).to_string())
    print(f"-> Mô hình chọn từ validation: {best_model[c]}, MAE test = {res.loc[best_model[c], 'MAE']:.4f}")

## 6. Kiểm tra không rò rỉ dữ liệu tương lai

Lấy ngẫu nhiên 300 mẫu của tập test cho mỗi chuỗi, tự tính lại bốn công thức từ chuỗi gốc rồi so với cột dự báo.
Seed lấy từ config nên chạy lại cho cùng kết quả.

In [ ]:
rng = np.random.default_rng(cfg["project"]["seed"])
for c in CHAINS:
    s = full_df[c].astype(float)
    pred = test_pred[c]
    idx = rng.choice(len(pred), size=300, replace=False)
    bad = 0
    for i in idx:
        ts = pred.loc[i, "timestamp"]
        p = s.index.get_loc(ts)
        expected = {
            "y_true": s.iloc[p],
            "y_last_value": s.iloc[p - 1],
            "y_ma3": s.iloc[p - 3:p].mean(),
            "y_snaive24": s.iloc[p - 24],
            "y_snaive168": s.iloc[p - 168],
        }
        bad += sum(not np.isclose(pred.loc[i, k], v) for k, v in expected.items())
    print(f"{c}: 300 mẫu x 5 cột, số giá trị sai = {bad}")
    assert bad == 0

## 7. Phân tích sai số của mô hình tốt nhất

Gom sai số theo giờ trong ngày (nhãn là mốc kết thúc giờ) và theo thứ trong tuần trên tập test, rồi liệt kê 10 dự báo sai nhiều nhất.
Mỗi mốc ngày chuyển giờ (DST) được kiểm tra xem có lọt vào top-10 hay không.

In [ ]:
err_hour, err_dow, top10 = {}, {}, {}
dst_set = {ts for ts, _ in dst_timestamps(cfg)}

for c in CHAINS:
    m = best_model[c]
    df = test_pred[c].copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])

    err_hour[c] = evaluate_by_group(df, pred_col=m, group_by="hour")
    err_dow[c] = evaluate_by_group(df, pred_col=m, group_by="dayofweek")

    df["abs_error"] = (df["y_true"] - df[m]).abs()
    top10[c] = df.sort_values("abs_error", ascending=False).head(10)

    h = err_hour[c].sort_values("MAE")
    print(f"\n=== {c} - mô hình {m} ===")
    print("Giờ (nhãn) sai nhiều nhất:", h.tail(3)["hour"].tolist()[::-1],
          "| ít nhất:", h.head(3)["hour"].tolist())
    print("MAE theo thứ (0=T2 ... 6=CN):", err_dow[c].set_index("dayofweek")["MAE"].round(2).to_dict())
    print("Số mẫu theo thứ:", err_dow[c].set_index("dayofweek")["N_samples"].to_dict())
    n_dst = int(top10[c]["timestamp"].isin(dst_set).sum())
    print(f"Mốc DST trong top-10 sai số lớn nhất: {n_dst}/10")
    display(top10[c].round(2))

**Chú thích về số mẫu theo thứ:** số mẫu không đều giữa các thứ vì tập test bắt đầu lúc 01:00 ngày 01/08/2014 và kết thúc lúc 00:00 ngày 01/01/2015,
nên hai ngày biên bị cắt bớt giờ. Đây không phải lỗi dữ liệu.

## 8. Biểu đồ

Ba biểu đồ cho mỗi chuỗi: so sánh MAE/RMSE bốn baseline trên validation, dự báo tuần đầu của test, và MAE theo giờ trong ngày.

In [ ]:
for c in CHAINS:
    m = best_model[c]

    ax = val_res[c][["MAE", "RMSE"]].plot(kind="bar", figsize=(9, 5))
    ax.set(title=f"{c}: MAE và RMSE của các baseline - Validation", xlabel="Baseline", ylabel="Sai số (kW)")
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(FIG_DIR / f"baseline_val_comparison_{c}.png", dpi=200); plt.show()

    n = min(168, len(test_pred[c]))
    ts = pd.to_datetime(test_pred[c]["timestamp"])[:n]
    plt.figure(figsize=(14, 5))
    plt.plot(ts, test_pred[c]["y_true"][:n], label="Thực tế", color="black", linewidth=1.5)
    plt.plot(ts, test_pred[c][m][:n], label=f"Dự báo: {m}", color="red", linestyle="--", linewidth=1.2)
    plt.title(f"{c}: dự báo tuần đầu của test - {m}")
    plt.xlabel("Thời gian"); plt.ylabel("Phụ tải (kW)"); plt.legend(); plt.tight_layout()
    plt.savefig(FIG_DIR / f"baseline_test_timeseries_{c}.png", dpi=200); plt.show()

    plt.figure(figsize=(9, 4))
    plt.bar(err_hour[c]["hour"], err_hour[c]["MAE"], color="skyblue")
    plt.title(f"{c}: MAE theo giờ trong ngày - {m}")
    plt.xlabel("Giờ trong ngày (nhãn 0-23)"); plt.ylabel("MAE (kW)"); plt.xticks(range(24)); plt.tight_layout()
    plt.savefig(FIG_DIR / f"baseline_error_by_hour_{c}.png", dpi=200); plt.show()

## 9. Lưu kết quả

Mọi tệp có hậu tố tên chuỗi, nên các chuỗi không ghi đè lên nhau.
`baseline_predictions_<chuỗi>.csv` gộp validation và test; `eval_timestamps_<chuỗi>.csv` là danh sách mốc đánh giá chung cho mô hình học sâu.

In [ ]:
saved = []
for c in CHAINS:
    v = val_pred[c].copy(); v.insert(0, "split", "validation")
    t = test_pred[c].copy(); t.insert(0, "split", "test")

    files = {
        TAB_DIR / f"baseline_results_{c}.csv": (val_res[c], True),
        TAB_DIR / f"baseline_test_results_{c}.csv": (test_res[c], True),
        TAB_DIR / f"error_by_hour_{c}.csv": (err_hour[c], False),
        TAB_DIR / f"error_by_dayofweek_{c}.csv": (err_dow[c], False),
        TAB_DIR / f"top10_worst_errors_{c}.csv": (top10[c], False),
        RES_DIR / f"baseline_predictions_validation_{c}.csv": (val_pred[c], False),
        RES_DIR / f"baseline_predictions_test_{c}.csv": (test_pred[c], False),
        RES_DIR / f"baseline_predictions_{c}.csv": (pd.concat([v, t], ignore_index=True), False),
        RES_DIR / f"eval_timestamps_{c}.csv": (pd.concat(
            [pd.DataFrame({"split": "validation", "timestamp": val_ts[c]}),
             pd.DataFrame({"split": "test", "timestamp": test_ts[c]})], ignore_index=True), False),
    }
    for path, (frame, keep_index) in files.items():
        frame.to_csv(path, index=keep_index)
        saved.append(path.relative_to(ROOT).as_posix())

print(f"Đã lưu {len(saved)} tệp kết quả + {len(CHAINS)} scaler trong models/:")
print("\n".join(saved))

## 10. Tổng hợp bốn chuỗi

Bảng này là nguyên liệu cho Chương 3 và cho lập luận chọn chuỗi: chuỗi có `ac_lag24` cao thì Seasonal Naive 24 giờ thắng,
chuỗi chính MT_158 (`ac_lag24` thấp) thì Last Value dẫn đầu và còn dư địa cho mô hình học sâu.

In [ ]:
rows = []
for c in CHAINS:
    r = val_res[c]["MAE"]
    rows.append({
        "customer": c,
        **{k: r[k] for k in BASELINE_COLUMNS},
        "best_val": best_model[c],
        "MAE_test_best": test_res[c].loc[best_model[c], "MAE"],
        "MASE_last_value_test": test_res[c].loc["y_last_value", "MASE"],
    })
summary = pd.DataFrame(rows).set_index("customer")
summary.to_csv(TAB_DIR / "baseline_summary_all_chains.csv")
summary.round(3)

## 11. Đối chiếu chéo với kết quả có sẵn

MT_168 đã được Đăng Long chạy riêng (`notebooks/results/MT_168/`). Cell dưới so MAE của lần chạy này với tệp đó:
hai lần chạy dùng cùng dữ liệu và cùng công thức nên phải trùng tới sai số làm tròn. Nếu không có tệp thì bỏ qua.

In [ ]:
ref = ROOT / "notebooks" / "results" / "MT_168" / "tables" / "baseline_results.csv"
if "MT_168" in CHAINS and ref.exists():
    other = pd.read_csv(ref).set_index("model")["MAE"]
    mine = val_res["MT_168"]["MAE"]
    diff = (mine - other.reindex(mine.index)).abs().max()
    print(f"MT_168: lệch MAE validation lớn nhất so với kết quả của Đăng Long = {diff:.2e}")
    assert diff < 1e-9
else:
    print("Không có tệp đối chiếu MT_168, bỏ qua.")

## 12. Nhận xét

_(nhận xét được điền sau khi chạy)_